# 03 Train Symbolic SODT

- `z`: the frozen RoI Align pooled feature grid for each RPN proposal.
- `teacher_label`: the original Faster R-CNN classifier decision on that same pooled RoI.
- `bbox regression`: stays neural and is not part of SODT training.

This notebook is the canonical symbolic stage of the thesis pipeline:
1. load the trained Faster R-CNN teacher,
2. export symbolic RoI datasets for `trainval` and `test`,
3. train one configured SODT with TAO on the exported `trainval` RoIs,
4. evaluate that trained SODT on the held-out `test` export.


In [1]:
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Markdown, display

from notebooks.util import resolve_root
from neuro.config import NeuroConfig, NeuroTrainConfig
from neuro.faster_rcnn import NeuroFasterRCNN
from neuro.prepare_dataset import PCBDataset
from neuro.preprocess_dataset import test_preprocess
from symbolic.config import SymbolicTrainConfig
from symbolic.export import export_teacher_roi_dataset
from symbolic.train import train_symbolic_tree
from util.artifacts import latest_run_checkpoint
from util.config import load_yaml
from util.device import select_device

PROJECT_ROOT = resolve_root()


In [2]:
train_config = load_yaml(Path("neuro_train.yaml"), NeuroTrainConfig)
neuro_config = load_yaml(Path("neuro.yaml"), NeuroConfig)
symbolic_train_config = load_yaml(Path("symbolic_train.yaml"), SymbolicTrainConfig)

device = select_device(train_config["device"])
checkpoint_dir = Path("checkpoints") / "neuro"
checkpoint_path = latest_run_checkpoint(checkpoint_dir)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)

model = NeuroFasterRCNN(
    neuro_config=neuro_config,
    train_config=train_config,
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

symbolic_dir = Path("checkpoints") / "symbolic"
teacher_trainval_path = symbolic_dir / "teacher_trainval.pt"
teacher_test_path = symbolic_dir / "teacher_test.pt"
symbolic_checkpoint_path = symbolic_dir / "sodt_run1.pt"
symbolic_summary_path = symbolic_dir / "sodt_run1_summary.json"

checkpoint_path, checkpoint.get("metrics", {})


(PosixPath('checkpoints/neuro/run2.pt'),
 {'mAP@0.5:0.95': 0.7576388120651245,
  'mAP@0.5': 0.9799945950508118,
  'AP75': 0.908657431602478,
  'AP@50:5:85': 0.8981851935386658,
  'precision': 0.9013754755633596,
  'recall': 0.9808917197452229,
  'mar_100': 0.8529934883117676,
  'precision_recall_score_threshold': 0.5,
  'per_class_AP': {'open': 0.6775085926055908,
   'short': 0.6452382206916809,
   'mouse_bite': 0.7480947375297546,
   'spur': 0.723054826259613,
   'pinhole': 0.8867635130882263,
   'spurious_copper': 0.865172803401947}})

In [3]:
trainval_dataset = PCBDataset(train_config, "trainval.txt")
trainval_dataset.add_preprocess(test_preprocess())

test_dataset = PCBDataset(train_config, "test.txt")
test_dataset.add_preprocess(test_preprocess())
def summarize_symbolic_export(export_path: Path, split_name: str):
    manifest = torch.load(export_path, map_location="cpu", weights_only=True)
    records = pd.DataFrame(manifest["records"])

    if records.empty:
        overview = pd.DataFrame(
            [
                {
                    "split": split_name,
                    "images": 0,
                    "total_rois": 0,
                    "background_rois": 0,
                    "positive_rois": 0,
                    "feature_shape": None,
                    "feature_cut": manifest.get("feature_cut"),
                    "symbolic_target": manifest.get("symbolic_target"),
                    "proposal_source": manifest.get("proposal_source"),
                }
            ]
        )
        class_counts = pd.DataFrame(columns=["split", "class_name", "count"])
        return manifest, overview, class_counts

    class_names = list(manifest["class_names"])
    label_counts = records["label_counts"].apply(pd.Series).fillna(0).sum(axis=0)
    label_counts = label_counts.reindex(range(len(class_names)), fill_value=0).astype(int)

    feature_shape = manifest.get("feature_shape")
    overview = pd.DataFrame(
        [
            {
                "split": split_name,
                "images": int(len(records)),
                "total_rois": int(records["num_rois"].sum()),
                "background_rois": int(label_counts.iloc[0]),
                "positive_rois": int(label_counts.iloc[1:].sum()),
                "feature_shape": "x".join(str(value) for value in feature_shape) if feature_shape else None,
                "feature_cut": manifest.get("feature_cut"),
                "symbolic_target": manifest.get("symbolic_target"),
                "proposal_source": manifest.get("proposal_source"),
            }
        ]
    )
    class_counts = pd.DataFrame(
        {
            "split": split_name,
            "class_name": class_names,
            "count": label_counts.tolist(),
        }
    )
    return manifest, overview, class_counts


len(trainval_dataset), len(test_dataset)


(1000, 500)

In [4]:
export_teacher_roi_dataset(
    model=model,
    dataset=trainval_dataset,
    output_path=teacher_trainval_path,
    device=train_config["device"],
    max_positive_rois_per_image=24,
    max_background_rois_per_image=40,
    storage_dtype="float16",
)
export_teacher_roi_dataset(
    model=model,
    dataset=test_dataset,
    output_path=teacher_test_path,
    device=train_config["device"],
    max_positive_rois_per_image=24,
    max_background_rois_per_image=40,
    storage_dtype="float16",
)

teacher_trainval_path, teacher_test_path


(PosixPath('checkpoints/symbolic/teacher_trainval.pt'),
 PosixPath('checkpoints/symbolic/teacher_test.pt'))

In [5]:
trainval_manifest, trainval_overview, trainval_counts = summarize_symbolic_export(
    teacher_trainval_path,
    "trainval",
)
test_manifest, test_overview, test_counts = summarize_symbolic_export(
    teacher_test_path,
    "test",
)

display(pd.concat([trainval_overview, test_overview], ignore_index=True))
display(pd.concat([trainval_counts, test_counts], ignore_index=True))


,split,images,total_rois,background_rois,positive_rois,feature_shape,feature_cut,symbolic_target,proposal_source
0,trainval,1000,63996,40000,23996,256x7x7,roi_align_pooled_grid,teacher_label,rpn_pre_detector_postprocess
1,test,500,32000,20000,12000,256x7x7,roi_align_pooled_grid,teacher_label,rpn_pre_detector_postprocess


,split,class_name,count
0,trainval,__background__,40000
1,trainval,open,3673
2,trainval,short,2484
3,trainval,mouse_bite,3928
4,trainval,spur,3499
5,trainval,pinhole,5936
6,trainval,spurious_copper,4476
7,test,__background__,20000
8,test,open,2055
9,test,short,1213


In [6]:
summary = train_symbolic_tree(
    export_path=teacher_trainval_path,
    output_path=symbolic_checkpoint_path,
    summary_path=symbolic_summary_path,
    heldout_export_path=teacher_test_path,
    config=symbolic_train_config,
)

train_report = pd.DataFrame([summary["train_report"]])

training_columns = [
    "model_id",
    "tree_depth",
    "l1_lambda",
    "sparsity_alpha",
    "mimic_accuracy",
    "macro_f1_vs_teacher",
    "nonzero_weights",
    "mean_path_feature_count",
    "box_grounded_roi_overlap",
    "pointing_score",
]

display(Markdown("### Trainval Training Metrics"))
display(train_report[training_columns])


Loaded 16000 symbolic samples with 12544 raw RoI-pooled feature dimensions.
TAO consumes the full RoI Align feature vector. Symbolic sparsity comes only from L1 regularization and the tree structure.
Training one configured SODT on the configured trainval RoI set: depth=5, lambda=30, alpha=0.25.
Workload check: 40 TAO iterations and up to 1240 LIBLINEAR node fits.
Progress line shows the TAO step, mimic, sparsity, and ETA in one place. No side logs are emitted while the bar is running.


Symbolic SODT training: 100%|██████████| 40/40 [26:49<00:00, 40.24s/it, d=5 lam=30 a=0.25 | tao 40/40 | mimic=1.0000 | nz=19129 | eta=0s]     


### Trainval Training Metrics

,model_id,tree_depth,l1_lambda,sparsity_alpha,mimic_accuracy,macro_f1_vs_teacher,nonzero_weights,mean_path_feature_count,box_grounded_roi_overlap,pointing_score
0,depth5_lambda30_alpha0.25,5,30.0,0.25,1.0,1.0,19129,4355.233562,0.89797,0.957625


In [7]:
model_summary = summary["model"]
model_metrics = pd.DataFrame(
    [
        {
            "model_id": model_summary["model_id"],
            "tree_depth": model_summary["tree_depth"],
            "l1_lambda": model_summary["l1_lambda"],
            "sparsity_alpha": model_summary["sparsity_alpha"],
            "train_mimic_accuracy": model_summary["metrics"]["mimic_accuracy"],
            "train_macro_f1_vs_teacher": model_summary["metrics"]["macro_f1_vs_teacher"],
            "nonzero_weights": model_summary["metrics"]["nonzero_weights"],
            "mean_path_feature_count": model_summary["metrics"]["mean_path_feature_count"],
            "box_grounded_roi_overlap": model_summary["metrics"]["box_grounded_roi_overlap"],
            "pointing_score": model_summary["metrics"]["pointing_score"],
        }
    ]
)

heldout_review = pd.DataFrame([summary["heldout_review"]["report"]])

display(Markdown("### Trained SODT Metrics"))
display(model_metrics)
display(Markdown("### Held-out Test Metrics"))
display(
    heldout_review[
        [
            "model_id",
            "tree_depth",
            "mimic_accuracy",
            "macro_f1_vs_teacher",
            "nonzero_weights",
            "mean_path_feature_count",
            "box_grounded_roi_overlap",
            "pointing_score",
        ]
    ]
)

{
    "symbolic_input": summary["training_config"]["symbolic_input"],
    "training_config": summary["training_config"],
    "heldout_review_note": summary["heldout_review"]["note"],
}


### Trained SODT Metrics

,model_id,tree_depth,l1_lambda,sparsity_alpha,train_mimic_accuracy,train_macro_f1_vs_teacher,nonzero_weights,mean_path_feature_count,box_grounded_roi_overlap,pointing_score
0,depth5_lambda30_alpha0.25,5,30.0,0.25,1.0,1.0,19129,4355.233562,0.89797,0.957625


### Held-out Test Metrics

,model_id,tree_depth,mimic_accuracy,macro_f1_vs_teacher,nonzero_weights,mean_path_feature_count,box_grounded_roi_overlap,pointing_score
0,depth5_lambda30_alpha0.25,5,0.980969,0.975004,19129,3054.474813,0.793643,0.824247


{'symbolic_input': 'raw_roi_align_pooled_grid',
 'training_config': {'tree_depth': 5,
  'iterations': 40,
  'l1_lambda': 30.0,
  'sparsity_alpha': 0.25,
  'logistic_max_iter': 250,
  'tolerance': 0.0001,
  'zero_threshold': 1e-05,
  'random_state': 42,
  'include_background': True,
  'min_teacher_score': None,
  'max_samples_total': 16000,
  'symbolic_input': 'raw_roi_align_pooled_grid',
  'sparsity_source': 'l1_regularization_and_tree_structure',
  'heldout_export_path': 'checkpoints/symbolic/teacher_test.pt',
  'config_sections': {'data': {'include_background': True,
    'min_teacher_score': None,
    'max_samples_total': 16000},
   'search': {'tree_depth': 5,
    'iterations': 40,
    'l1_lambda': 30,
    'sparsity_alpha': 0.25,
    'logistic_max_iter': 250,
    'tolerance': 0.0001,
    'zero_threshold': 1e-05,
    'random_state': 42}}},
 'heldout_review_note': 'Held-out test evaluates the trained SODT once. It does not participate in training.'}